## ????? ??? 

In [ ]:
## library import - data preprocessing
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder

In [ ]:
## BERT embedding 
from transformers import BertTokenizer, BertModel
import torch 

In [ ]:
## Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
## ?? ?? 
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import spearmanr

In [ ]:
## visualization
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
import matplotlib.font_manager as fm

# ?? ?? ?? (????)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # ???? ?? ?? ??

In [ ]:
# tqdm 
from tqdm import tqdm 

In [ ]:
# kaggle hub
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
## GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("?? ?? ????:", device)

In [ ]:
print("CUDA ?? ???", torch.cuda.is_available())
if torch.cuda.is_available():
    print("?? ?? GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU? ??? ? ????.")


## data load

In [ ]:
pip install kagglehub[pandas-datasets]

In [ ]:
# Download latest version
path = kagglehub.dataset_download("datasnaek/youtube-new")

print("Path to dataset files:", path)

In [ ]:
# load USvideo Dataset
df = pd.read_csv(os.path.join(path, "USvideos.csv"))

In [ ]:
# data ??
df.head()

In [ ]:
# ??? ?? 
df.info()

## data prepocessing

### Null/duplicate check

In [ ]:
df.isnull().sum()

In [ ]:
# ?? ?? 
df.duplicated().sum()

In [ ]:
# ??? ? ?? 
df[df.duplicated()]

### category mapping

In [ ]:
## ???? ?? 
# JSON ?? ??
json_path = os.path.join(path, "US_category_id.json")

# JSON ??
with open(json_path, 'r', encoding='utf-8') as f:
    category_json = json.load(f)

# ???? ?? ???? ??
category_mapping = {
    int(item['id']): item['snippet']['title']
    for item in category_json['items']
}

In [ ]:
# ?? ???? ??
print(category_mapping)

In [ ]:
df['category_name'] = df['category_id'].map(category_mapping)

In [ ]:
df['category_name']

### target log-scaling

In [ ]:
# log-scaling ??
df['log_views'] = np.log1p(df['views'])

## Modeling #1. category_id? feature? ?? 

In [ ]:
# category? ??? ? ???? 
category_df = df[['title', 'log_views', 'category_id']]
category_df

In [ ]:
# BERT load 
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
# BERT embedding extract 

## CLS ?? ?? ?? ?? 

def extract_cls_embedding(text):
    encoding = tokenizer(text,
                         max_length=32,
                         truncation=True,
                         padding='max_length',
                         return_tensors='pt')
    input_ids_tensor = encoding['input_ids'].to(device)
    attention_mask_tensor = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return cls_embedding


In [ ]:
## ?? ??? cls? ?? 
embeddings = np.array([
    extract_cls_embedding(title) for title in tqdm(category_df['title'])
])

In [ ]:
# category one-hot encoding

ohe = OneHotEncoder(sparse_output=False)
X_category = ohe.fit_transform(category_df[['category_id']])

In [ ]:
# ? ?? ??
X = np.hstack([embeddings, X_category])

# target ??
y = category_df['log_views']

# test, train dataset ?? 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

In [ ]:
# Model ?? ?? ?? 
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    spearman_corr, _ = spearmanr(y_test, y_pred)

    print(f"RMSE: {rmse:.4f}")
    print(f"R?: {r2:.4f}")
    print(f"Spearman: {spearman_corr:.4f}")
    
    return model, rmse, r2, spearman_corr

In [ ]:
# Linear Regression 
lr = LinearRegression()
evaluate_model(lr, X_train, y_train, X_test, y_test)

In [ ]:
ridge = Ridge(alpha=1.0)
evaluate_model(ridge, X_train, y_train, X_test, y_test)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
evaluate_model(rf, X_train, y_train, X_test, y_test)

In [ ]:
gradient = GradientBoostingRegressor(n_estimators=100, random_state=42)
evaluate_model(gradient, X_train, y_train, X_test, y_test)

## Modeling#2 category? Model ?? 

### category ?? 

In [ ]:
# category? ??? ? ?? 
category_counts = df['category_name'].value_counts().sort_values(ascending=True)

In [ ]:
plt.figure(figsize=(10, 6))
category_counts.plot(kind='barh', color='skyblue')

plt.title("????? ??? ?? ?", fontsize=14)
plt.xlabel("?? ?", fontsize=12)
plt.ylabel("category_id", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# BERT embedding? category ??
category_df['title_embedding'] = list(embeddings)

### Entertainment

In [ ]:
# category filtering
cat_name = "Entertainment"
group = category_df[category_df['category_id'] == 24]

X = np.vstack(group['title_embedding'].values)
y = group['log_views']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

#### Entertainment - Lasso

In [ ]:
# Lasso ?? ??
alpha_value = 0.0001
lasso = Lasso(alpha=alpha_value)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_test)

# ?? ?? ??
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
spearman_corr = spearmanr(y_test, y_pred).correlation if np.std(y_pred) > 0 else np.nan

# ??
print("Category:", cat_name)
print("Model: Lasso")
print("Alpha:", alpha_value)
print("RMSE:", round(rmse, 4))
print("R?:", round(r2, 4))
print("Spearman:", "NaN" if np.isnan(spearman_corr) else round(spearman_corr, 4))

# ?? ?? 
lasso_results_df = pd.DataFrame([{
    "Category": cat_name,
    "Model": "Lasso",
    "Alpha": alpha_value,
    "RMSE": rmse,
    "R?": r2,
    "Spearman": spearman_corr
}])

#### Entertainment - Linear Regression

In [ ]:
# LinearRegression ?? ?? ? ??
linear = LinearRegression()
linear.fit(X_train, y_train)
y_pred = linear.predict(X_test)

# ?? ?? ??
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
spearman_corr = spearmanr(y_test, y_pred).correlation if np.std(y_pred) > 0 else np.nan

# ??
print("Category:", cat_name)
print("Model: LinearRegression")
print("RMSE:", round(rmse, 4))
print("R?:", round(r2, 4))
print("Spearman:", "NaN" if np.isnan(spearman_corr) else round(spearman_corr, 4))

# ?? ??
linear_results_df = pd.DataFrame([{
    "Category": cat_name,
    "Model": "LinearRegression",
    "Alpha": None,
    "RMSE": rmse,
    "R?": r2,
    "Spearman": spearman_corr
}])

#### Entertainment - Ridge

In [ ]:
# Ridge ?? ?? ? ??
alpha_value = 1.0
ridge = Ridge(alpha=alpha_value)
ridge.fit(X_train, y_train)
y_pred = ridge.predict(X_test)

# ?? ?? ??
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
spearman_corr = spearmanr(y_test, y_pred).correlation if np.std(y_pred) > 0 else np.nan

# ??
print("Category:", cat_name)
print("Model: Ridge")
print("Alpha:", alpha_value)
print("RMSE:", round(rmse, 4))
print("R?:", round(r2, 4))
print("Spearman:", "NaN" if np.isnan(spearman_corr) else round(spearman_corr, 4))

# ?? ??
ridge_results_df = pd.DataFrame([{
    "Category": cat_name,
    "Model": "Ridge",
    "Alpha": alpha_value,
    "RMSE": rmse,
    "R?": r2,
    "Spearman": spearman_corr
}])


#### Entertainment - RF 

In [ ]:
# RF ?? ?? ? ??
n_estimators = 100
rf = RandomForestRegressor(n_estimators=n_estimators, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# ?? ?? ??
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
spearman_corr = spearmanr(y_test, y_pred).correlation if np.std(y_pred) > 0 else np.nan

# ??
print("Category:", cat_name)
print("Model: RandomForest")
print("n_estimators:", n_estimators)
print("RMSE:", round(rmse, 4))
print("R?:", round(r2, 4))
print("Spearman:", "NaN" if np.isnan(spearman_corr) else round(spearman_corr, 4))

# ?? ??
rf_results_df = pd.DataFrame([{
    "Category": cat_name,
    "Model": "RandomForest",
    "Alpha": None,
    "RMSE": rmse,
    "R?": r2,
    "Spearman": spearman_corr
}])


#### Entertainment - Gradient 

In [ ]:
# Gradient Boosting ?? ?? ? ??
n_estimators = 100
gbr = GradientBoostingRegressor(n_estimators=n_estimators, random_state=42)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)

# ?? ?? ??
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
spearman_corr = spearmanr(y_test, y_pred).correlation if np.std(y_pred) > 0 else np.nan

# ??
print("Category:", cat_name)
print("Model: GradientBoosting")
print("n_estimators:", n_estimators)
print("RMSE:", round(rmse, 4))
print("R?:", round(r2, 4))
print("Spearman:", "NaN" if np.isnan(spearman_corr) else round(spearman_corr, 4))

# ?? ??
gbr_results_df = pd.DataFrame([{
    "Category": cat_name,
    "Model": "GradientBoosting",
    "Alpha": None,
    "RMSE": rmse,
    "R?": r2,
    "Spearman": spearman_corr
}])


In [ ]:
# ??? ?? DataFrame ???
all_results = pd.concat([
    lasso_results_df,
    linear_results_df,
    ridge_results_df,
    rf_results_df,
    gbr_results_df
], ignore_index=True)


In [ ]:
# ???? ?? ?? (wide ? long)
results_long = all_results.melt(
    id_vars="Model", 
    value_vars=["RMSE", "R?", "Spearman"],
    var_name="Metric", 
    value_name="Score"
)

# ? ???(line chart)? ???
plt.figure(figsize=(10, 6))
for metric in results_long["Metric"].unique():
    subset = results_long[results_long["Metric"] == metric]
    plt.plot(subset["Model"], subset["Score"], marker='o', label=metric)

plt.title("Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.legend(title="Metric")
plt.grid(True)
plt.tight_layout()
plt.show()